# EXP-005 Jev 리랭커 시험 호출

목적: Jev(`typesafe/jev-1.13`, OpenRouter Decisions API)를 리랭커로 쓸 때 요청 단위(질의당 한 요청에 후보 20개를 묶음, 쌍마다 한 요청), 과금 토큰, 지연, 반복 일치를 dev-small 질의 50개로 잰다. 번역 검토의 첫 단계로 지시문·등급 기준만 영어로 바꾼 조건도 함께 본다. 설계는 비용·지연·동작으로 고르고, 후보 안 정답 순위는 참고로만 본다(dev-small). 질의 원문은 출력하지 않는다(D-10). dev 질의의 OpenRouter 전송은 사용자 승인(2026-10-12).

In [6]:
import hashlib
import json
import os
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from itertools import groupby
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from rag.eval.metrics import load_cfg, load_gold

In [7]:
ROOT = Path.cwd().parent
PAIRS = ROOT / "data/rerank/summary_q-dev-small.pairs.jsonl"
OUT = ROOT / "data/rerank/jev_pilot"
URL = "https://openrouter.ai/api/alpha/decisions"
MODEL = "typesafe/jev-1.13"
N_QUERIES = 50
N_REPEAT = 10  # 반복 일치를 볼 질의 수
SEED = 20261012
RUBRIC = {
    "ko": {
        "instructions": "이 후보 문서는 질의에 얼마나 답하는가?",
        "levels": [
            "질의와 관련 없는 내용이다",
            "질의의 주제와 관련은 있지만 질의가 묻는 것에 답하지 않는다",
            "질의가 묻는 것의 일부를 간접적으로 담고 있다",
            "질의가 묻는 것의 대부분에 답한다",
            "질의가 묻는 것에 직접적이고 완전하게 답한다",
        ],
    },
    "en": {
        "instructions": "How well does this candidate document answer the query?",
        "levels": [
            "Unrelated to the query",
            "On the query's topic but does not answer what it asks",
            "Partly and indirectly contains what the query asks",
            "Answers most of what the query asks",
            "Directly and completely answers what the query asks",
        ],
    },
}

## 헬퍼

응답은 요청 본문의 해시로 저장해 다시 실행할 때 재사용한다. `repeat` 인자가 다르면 같은 본문이라도 새로 호출한다.

In [8]:
load_dotenv(ROOT / ".env")
KEY = os.environ["OPENROUTER_API"]
OUT.mkdir(parents=True, exist_ok=True)


def call(body: dict, repeat: int = 0) -> dict:
    raw = json.dumps(body, ensure_ascii=False, sort_keys=True)
    path = OUT / f"{hashlib.sha1(raw.encode()).hexdigest()[:16]}-{repeat}.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    req = urllib.request.Request(URL, data=raw.encode(), headers={
        "Authorization": f"Bearer {KEY}", "Content-Type": "application/json"})
    start = time.perf_counter()
    with urllib.request.urlopen(req, timeout=120) as r:
        resp = json.loads(r.read().decode())
    rec = {"latency": time.perf_counter() - start, "response": resp}
    path.write_text(json.dumps(rec, ensure_ascii=False), encoding="utf-8")
    return rec


def question(lang: str, ref: str) -> dict:
    r = RUBRIC[lang]
    return {"type": "score", "instructions": f"{ref}: {r['instructions']}", "criteria": r["levels"]}


def batched(group: list[dict], lang: str, repeat: int = 0) -> tuple[dict, dict]:
    """질의 하나에 후보를 모두 묶어 한 요청. 후보마다 질문 하나."""
    ref = "후보" if lang == "ko" else "candidate"
    cands = {str(i): p["text"] for i, p in enumerate(group)}
    state = {"query": group[0]["query"], "candidates": cands}
    qs = {f"c{i}": question(lang, f"{ref} {i}") for i in range(len(group))}
    rec = call({"model": MODEL, "state": state, "questions": qs}, repeat)
    ans = rec["response"]["answers"]
    return {p["doc_id"]: ans[f"c{i}"]["score"] for i, p in enumerate(group)}, rec


def per_pair(group: list[dict], lang: str) -> tuple[dict, list[dict]]:
    """쌍마다 한 요청을 병렬로 보낸다."""
    ref = "문서" if lang == "ko" else "document"
    bodies = [{"model": MODEL, "state": {"query": p["query"], "document": p["text"]},
               "questions": {"rel": question(lang, ref)}} for p in group]
    start = time.perf_counter()
    with ThreadPoolExecutor(len(bodies)) as ex:
        recs = list(ex.map(call, bodies))
    wall = time.perf_counter() - start
    scores = {p["doc_id"]: r["response"]["answers"]["rel"]["score"]
              for p, r in zip(group, recs, strict=True)}
    return scores, recs, wall

## 시험 질의

dev-small 후보 쌍 파일(질의당 RRF 상위 20개, 문서별 최고점 청크)에서 질의 50개를 고른다.

In [9]:
with PAIRS.open(encoding="utf-8") as f:
    groups = [list(g) for _, g in groupby(map(json.loads, f), key=lambda p: p["qid"])]
rng = np.random.default_rng(SEED)
picked = [groups[i] for i in sorted(rng.choice(len(groups), N_QUERIES, replace=False))]
print("질의", len(picked), "질의당 후보", {len(g) for g in picked})

질의 50 질의당 후보 {20}


## 과금 확인

질의 하나를 묶음 요청으로 보내 입력 토큰을 본다. 공유 `state`가 질문마다 과금되면 입력 토큰이 후보 본문 합의 약 20배가 된다.

In [10]:
s, rec = batched(picked[0], "ko")
u = rec["response"]["usage"]
chars = sum(len(p["text"]) for p in picked[0])
print("입력 토큰", u["input_tokens"], "비용", u.get("cost"), "지연", round(rec["latency"], 2), "초",
      "| 후보 본문 합", chars, "자")

입력 토큰 15279 비용 0.000641718 지연 0.51 초 | 후보 본문 합 16763 자


## 세 조건 실행

- `batch_ko`: 질의당 한 요청, 한국어 지시문·등급
- `pair_ko`: 쌍마다 한 요청(20개 병렬), 한국어
- `batch_en`: 질의당 한 요청, 지시문·등급만 영어(질의·문서는 한국어)

지연은 묶음은 요청 하나, 쌍별은 20개 병렬 요청의 전체 시간이다.

In [11]:
runs, usage = {}, []
for g in picked:
    q = g[0]["qid"]
    for lang in ("ko", "en"):
        s, rec = batched(g, lang)
        runs.setdefault(f"batch_{lang}", {})[q] = s
        u = rec["response"]["usage"]
        usage.append({"cond": f"batch_{lang}", "qid": q, "tokens": u["input_tokens"],
                      "cost": u.get("cost") or 0, "latency": rec["latency"]})
    s, recs, wall = per_pair(g, "ko")
    runs.setdefault("pair_ko", {})[q] = s
    usage.append({"cond": "pair_ko", "qid": q,
                  "tokens": sum(r["response"]["usage"]["input_tokens"] for r in recs),
                  "cost": sum(r["response"]["usage"].get("cost") or 0 for r in recs),
                  "latency": wall})
usage = pd.DataFrame(usage)
usage.groupby("cond").agg(tokens=("tokens", "mean"), cost=("cost", "sum"),
                          latency_p50=("latency", "median"),
                          latency_p95=("latency", lambda x: x.quantile(0.95))).round(4)

,tokens,cost,latency_p50,latency_p95
cond,,,,
batch_en,15091.72,0.0317,0.5160,1.1785
batch_ko,16031.72,0.0337,0.5123,0.7428
pair_ko,21937.70,0.0461,0.7463,2.2908


## 반복 일치

`batch_ko`를 질의 10개에 다시 보내 점수 차이를 본다.

In [12]:
diffs = []
for g in picked[:N_REPEAT]:
    s2, _ = batched(g, "ko", repeat=1)
    s1 = runs["batch_ko"][g[0]["qid"]]
    diffs += [abs(s1[d] - s2[d]) for d in s1]
print("점수 차이 평균", round(float(np.mean(diffs)), 4), "최대", round(float(np.max(diffs)), 4))

점수 차이 평균 0.0404 최대 0.66


## 후보 안 정답 순위(참고)

후보 20개 안에서 정답 문서의 순위를 RRF 순서와 비교한다. dev-small이고 질의 50개라 설계 선택에는 쓰지 않는다. 점수 동점은 RRF 순서로 정렬한다(실제 재정렬과 같음).

In [13]:
gold = load_gold(load_cfg(), "dev-small")["doc_id"]


def gold_rank(order: list[str], q: str):
    return order.index(gold[q]) + 1 if gold[q] in order else None


rows = []
for g in picked:
    q = g[0]["qid"]
    rrf_order = [p["doc_id"] for p in g]
    row = {"qid": q, "rrf": gold_rank(rrf_order, q)}
    for name, sc in runs.items():
        row[name] = gold_rank(sorted(rrf_order, key=lambda d: -sc[q][d]), q)
        row[f"{name}_distinct"] = len(set(sc[q].values()))
    rows.append(row)
ranks = pd.DataFrame(rows).set_index("qid")
conds = ["rrf", *runs]
print("후보 20개 안에 정답이 있는 질의", int(ranks["rrf"].notna().sum()), "/", len(ranks))
has = ranks[ranks["rrf"].notna()]
pd.DataFrame({
    "hit@1": [(has[c] == 1).mean() for c in conds],
    "hit@5": [(has[c] <= 5).mean() for c in conds],
    "mrr": [(1 / has[c]).mean() for c in conds],
}, index=conds).round(3)

후보 20개 안에 정답이 있는 질의 50 / 50


,hit@1,hit@5,mrr
rrf,0.94,1.00,0.970
batch_ko,0.88,1.00,0.925
batch_en,0.88,1.00,0.926
pair_ko,0.72,0.94,0.828


In [14]:
print("질의당 서로 다른 점수 수(20개 중) 중앙값",
      {n: float(ranks[f"{n}_distinct"].median()) for n in runs})

질의당 서로 다른 점수 수(20개 중) 중앙값 {'batch_ko': 18.5, 'batch_en': 18.0, 'pair_ko': 19.0}
